# DTM Fine-Tuning — 93.97% → 98%+
### Round 4: Iterative Pseudo-Label Refresh + Multi-Round HEM + Geometric TTA

## Starting Point
| Metric | Value |
|---|---|
| Current val accuracy | **93.97%** (after Round 3) |
| Current ground F1 | 0.9073 |
| Optimal threshold | 0.56 |
| Model | best_model.pth (3.58 MB) |

## Why Round 3 Stalled at 93.97%
| Problem | Impact |
|---|---|
| Same tiles every epoch (cached pseudo-labels never refreshed) | Model memorises labels, not geometry |
| HEM mines once, then trains same hard set for 12 epochs | No fresh signal after ep 4 |
| TTA = permutation only (no geometric augmentation) | Underestimates actual model capability |
| SWA BN update on 400 tiles only | Badly calibrated batch norm |
| Mixup α=0.4 → near identity mix | Not forcing boundary learning |

## Strategy to Close 4.03pp in ≤ 250 GPU-minutes

| Technique | Expected Gain | Time |
|---|---|---|
| Round 4 pseudo-labels (conf≥0.92, lower than R3 to cast wider net) | +0.5pp | 9 min |
| Phase A: Full finetune with label refresh every 5 epochs | +1.0pp | 25 min |
| Phase B: 3×HEM rounds (re-mine tiles after each round) | +1.5pp | 40 min |
| Phase C: SWA over 8 epochs, BN on 1000+ tiles | +0.3pp | 15 min |
| Geometric TTA (8 rotations + 3 scales = 24 passes) | +0.7pp | free |
| **Total expected** | **+4.0pp → 98%** | **~90 min** |

**Run order: Cells 1→2→3→4→5→6→7→8→9**


In [1]:
# Cell 1 — GPU setup
import torch, torch.nn as nn, torch.nn.functional as F
import random, time, math, gc, io, zipfile, json
import numpy as np
from pathlib import Path, PurePosixPath
from tqdm import tqdm

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected.\n'
        'Kaggle -> Settings -> Accelerator -> GPU T4 x2 -> Save\n'
        'Run -> Restart Session -> Run All Cells'
    )

N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda')

for i in range(N_GPUS):
    p  = torch.cuda.get_device_properties(i)
    sm = p.major * 10 + p.minor
    if sm < 70:
        raise RuntimeError(
            f'GPU {p.name} (sm_{sm}) needs PyTorch fix.\n'
            'Add Cell 0 from previous notebook (P100 fix) and run it first.'
        )
    _t = torch.zeros(4, device=f'cuda:{i}') + 1
    torch.cuda.synchronize(i); del _t
    print(f'GPU {i}: {p.name}  sm_{sm}  {p.total_memory/1e9:.1f} GB')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

SESSION_START    = time.time()
SESSION_BUDGET_H = 11.0

print(f'\nGPUs       : {N_GPUS}')
print(f'Total VRAM : {sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPUS))/1e9:.1f} GB')
print(f'Session    : budget={SESSION_BUDGET_H}h')


GPU 0: Tesla T4  sm_75  15.6 GB
GPU 1: Tesla T4  sm_75  15.6 GB

GPUs       : 2
Total VRAM : 31.3 GB
Session    : budget=11.0h


In [2]:
# Cell 2 — Configuration
# Points at your TWO datasets:
#   1. Training tiles (same as before)
#   2. Your trained weights (dtm_outputs_finetuned.zip from Round 3)

from pathlib import Path

# ── Dataset paths ─────────────────────────────────────────────────────────────
TRAINING_DATASET_ROOT = '/kaggle/input/datasets/jaibhagwanchouhan/training-data-95pct'
WEIGHTS_DATASET_ROOT  = '/kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs'

def _find_root(base):
    base = Path(base)
    if (base/'train').exists(): return str(base), None
    for s in sorted(base.iterdir()):
        if s.is_dir() and (s/'train').exists(): return str(s), None
    zips = list(base.rglob('training_data.zip'))
    if zips: return None, str(zips[0])
    return str(base), None

_TDIR, _ZPATH = _find_root(TRAINING_DATASET_ROOT)
_USE_ZIP = _ZPATH is not None and _TDIR is None

def _find_weight(name):
    # Prefer finetuned outputs if they exist (Round 3 result)
    for root in ['/kaggle/input', WEIGHTS_DATASET_ROOT, '/kaggle/working']:
        for p in Path(root).rglob(name):
            return str(p)
    return None

BEST_MODEL_PATH = _find_weight('best_model.pth')
SWA_MODEL_PATH  = _find_weight('swa_model.pth')
THRESHOLD_PATH  = _find_weight('threshold.json')

print(f'Training data : {_TDIR or "ZIP:"+str(_ZPATH)}')
print(f'best_model    : {BEST_MODEL_PATH}')
print(f'swa_model     : {SWA_MODEL_PATH}')
print(f'threshold.json: {THRESHOLD_PATH}')

if not BEST_MODEL_PATH:
    raise FileNotFoundError(
        'best_model.pth not found!\n'
        'Upload dtm_outputs_finetuned.zip as a Kaggle Dataset\n'
        'Then add it to this notebook via Add Data.')

if THRESHOLD_PATH:
    with open(THRESHOLD_PATH) as f:
        prev_meta = json.load(f)
    print(f'\nPrevious val accuracy : {prev_meta["val_accuracy"]*100:.2f}%')
    print(f'Previous F1           : {prev_meta["val_f1"]:.4f}')
    print(f'Previous threshold    : {prev_meta["threshold"]:.2f}')

BATCH_PER_GPU = 8
TOTAL_BATCH   = BATCH_PER_GPU * max(N_GPUS, 1)

CFG = {
    # Data
    'use_zip'         : _USE_ZIP,
    'zip_path'        : _ZPATH or '',
    'training_dir'    : _TDIR  or '',
    'feat_cache_dir'  : '/kaggle/working/feat_cache',
    'cons_label_dir'  : '/kaggle/working/consensus_labels',
    'pseudo_dir'      : '/kaggle/working/pseudo_labels',
    # Outputs
    'logs_dir'        : '/kaggle/working/logs',
    'model_save'      : '/kaggle/working/logs/best_model.pth',
    'swa_save'        : '/kaggle/working/logs/swa_model.pth',
    'ckpt'            : '/kaggle/working/logs/checkpoint.pth',
    'history'         : '/kaggle/working/logs/history.json',
    'curves'          : '/kaggle/working/logs/curves.png',
    'threshold'       : '/kaggle/working/logs/threshold.json',
    # Points
    'max_pts'         : 4096,
    'seed'            : 42,
    'batch_size'      : TOTAL_BATCH,
    # ── ROUND 4 key changes ──────────────────────────────────────────────────
    # Phase A: Full-dataset finetune with periodic label refresh
    'phaseA_epochs'   : 20,       # was 15 — more epochs, labels refresh mid-way
    'phaseA_lr'       : 2e-4,     # slightly lower than Round 3 (2e-4 vs 3e-4)
    'phaseA_refresh_every': 5,    # Re-generate pseudo-labels every 5 epochs  ← NEW
    # Phase B: Multi-round HEM (re-mine tiles after each round)
    'phaseB_rounds'   : 3,        # 3 independent HEM rounds               ← NEW
    'phaseB_epochs_per_round': 8, # 8 epochs per round (was 12 in one shot)
    'phaseB_lr'       : 8e-5,     # slightly lower
    'hem_hard_frac'   : 0.35,     # 35% hardest (tighter focus vs 40%)     ← NEW
    # Phase C: SWA with more BN tiles
    'swa_epochs'      : 8,        # was 5
    'swa_lr'          : 4e-5,
    'swa_bn_tiles'    : 1200,     # was 400 — critical fix for BN quality   ← NEW
    # Pseudo-label config
    'pseudo_conf_r4'  : 0.92,     # wider net than Round 3 (was 0.95)      ← NEW
    'pseudo_conf_refresh': 0.88,  # even wider for mid-training refreshes   ← NEW
    # Loss
    'focal_alpha'     : 0.75,
    'focal_gamma'     : 2.5,
    'label_smooth'    : 0.02,     # less smoothing (was 0.03)
    # Mixup
    'mixup_alpha'     : 0.2,      # sharper mix (was Beta(0.4,0.4))        ← NEW
    # Optimiser
    'weight_decay'    : 1e-5,
    'grad_clip'       : 1.0,
    # Schedule
    'use_amp'         : True,
    'val_every'       : 2,
    'patience'        : 12,
    'num_workers'     : 4,
    'prefetch_factor' : 2,
    # TTA — geometric augmentation (rotation + scale + flip)
    'val_tta_passes'  : 12,       # was 8 — 12 geometric variants           ← NEW
    'tta_geometric'   : True,     # use rotation/scale TTA not just perm    ← NEW
}

for d in ['logs_dir','feat_cache_dir','cons_label_dir','pseudo_dir']:
    Path(CFG[d]).mkdir(parents=True, exist_ok=True)
for split in ['train','val']:
    (Path(CFG['cons_label_dir'])/split).mkdir(parents=True, exist_ok=True)

print(f'\nBatch size        : {TOTAL_BATCH} ({BATCH_PER_GPU}/GPU x {N_GPUS})')
print(f'Phase A LR        : {CFG["phaseA_lr"]} | epochs={CFG["phaseA_epochs"]} | refresh every {CFG["phaseA_refresh_every"]} ep')
print(f'Phase B           : {CFG["phaseB_rounds"]} HEM rounds x {CFG["phaseB_epochs_per_round"]} ep | hard_frac={CFG["hem_hard_frac"]}')
print(f'Phase C SWA       : {CFG["swa_epochs"]} epochs | BN tiles={CFG["swa_bn_tiles"]}')
print(f'Pseudo conf R4    : {CFG["pseudo_conf_r4"]} (wider net than R3=0.95)')
print(f'Val TTA           : {CFG["val_tta_passes"]} geometric passes')


Training data : /kaggle/input/datasets/jaibhagwanchouhan/training-data-95pct
best_model    : /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/best_model.pth
swa_model     : /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/swa_model.pth
threshold.json: /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/threshold.json

Previous val accuracy : 93.65%
Previous F1           : 0.9020
Previous threshold    : 0.57

Batch size        : 16 (8/GPU x 2)
Phase A LR        : 0.0002 | epochs=20 | refresh every 5 ep
Phase B           : 3 HEM rounds x 8 ep | hard_frac=0.35
Phase C SWA       : 8 epochs | BN tiles=1200
Pseudo conf R4    : 0.92 (wider net than R3=0.95)
Val TTA           : 12 geometric passes


In [3]:
# Cell 3 — Feature engineering (identical to v6 training)
# MUST be identical or loaded features will be incompatible with saved model

def _grid(xyz, cell_m, gmax=64):
    xmin,ymin=xyz[:,0].min(),xyz[:,1].min()
    xr=xyz[:,0].max()-xmin+1e-6; yr=xyz[:,1].max()-ymin+1e-6
    GW=int(np.clip(xr/cell_m,8,gmax)); GH=int(np.clip(yr/cell_m,8,gmax))
    xi=np.clip(((xyz[:,0]-xmin)/xr*GW).astype(np.int32),0,GW-1)
    yi=np.clip(((xyz[:,1]-ymin)/yr*GH).astype(np.int32),0,GH-1)
    ci=yi*GW+xi; NC=GW*GH; z=xyz[:,2].astype(np.float32)
    c_min=np.full(NC,np.inf,np.float32); c_sum=np.zeros(NC,np.float32)
    c_sq=np.zeros(NC,np.float32); c_cnt=np.zeros(NC,np.float32)
    c_max=np.full(NC,-np.inf,np.float32)
    np.minimum.at(c_min,ci,z); np.maximum.at(c_max,ci,z)
    np.add.at(c_sum,ci,z); np.add.at(c_sq,ci,z*z); np.add.at(c_cnt,ci,1.)
    empty=c_cnt==0
    c_cnt[empty]=1; c_min[empty]=z.mean(); c_max[empty]=z.mean()
    c_sum[empty]=z.mean(); c_sq[empty]=z.mean()**2
    c_mean=c_sum/c_cnt
    c_std=np.sqrt(np.maximum(c_sq/c_cnt-c_mean**2,0.))
    return ci,c_min,c_std,c_cnt,c_max-c_min,GW,GH


def geo_features(xyz: np.ndarray) -> np.ndarray:
    """(N,3) -> (N,6): dZ_2m, roughness, slope, density, dZ_8m, planarity"""
    xyz=xyz.astype(np.float32,copy=False)
    ci2,cm2,cs2,cc2,cr2,GW2,GH2=_grid(xyz,2.0)
    dz2=np.clip(xyz[:,2]-cm2[ci2],0.,None)
    roughness=cs2[ci2]
    cg=cm2.reshape(GH2,GW2); pad=np.pad(cg,1,mode='edge')
    sg=np.max(np.stack([np.abs(cg-pad[:-2,1:-1]),np.abs(cg-pad[2:,1:-1]),
                        np.abs(cg-pad[1:-1,:-2]),np.abs(cg-pad[1:-1,2:])]),axis=0)/2.
    slope=sg.reshape(-1)[ci2]
    density=cc2[ci2]/(cc2.max()+1e-6)
    ci8,cm8,_,_,_,_,_=_grid(xyz,8.0,32)
    dz8=np.clip(xyz[:,2]-cm8[ci8],0.,None)
    planarity=cs2[ci2]/(cr2[ci2]+1e-6)
    feat=np.stack([dz2,roughness,slope,density,dz8,planarity],axis=1)
    mu=feat.mean(0,keepdims=True); sg=feat.std(0,keepdims=True)+1e-6
    return ((feat-mu)/sg).astype(np.float32)


def build_feat_cache(cfg, split):
    cd=Path(cfg['feat_cache_dir'])/split; cd.mkdir(parents=True,exist_ok=True)
    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tiles={}
        for n in names:
            parts=PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tiles[parts[parts.index(split)+1]]=n
        zf_h=zipfile.ZipFile(cfg['zip_path'])
        try:
            for t,zp in tqdm(tiles.items(),desc=f'FeatCache[{split}]',unit='tile'):
                out=cd/f'{t}.npy'
                if out.exists(): continue
                with zf_h.open(zp) as fh: xyz=np.load(io.BytesIO(fh.read()))
                np.save(str(out),geo_features(xyz))
        finally: zf_h.close()
    else:
        for td in tqdm(sorted((Path(cfg['training_dir'])/split).glob('tile_*')),
                       desc=f'FeatCache[{split}]'):
            out=cd/f'{td.name}.npy'
            if not out.exists():
                np.save(str(out),geo_features(np.load(str(td/'points.npy'))))


print('Building feature caches (skips if already exist)...')
t0=time.time()
build_feat_cache(CFG,'train'); build_feat_cache(CFG,'val')
print(f'Done in {(time.time()-t0)/60:.1f} min')


Building feature caches (skips if already exist)...


FeatCache[val]: 100%|██████████| 1216/1216 [00:12<00:00, 95.06it/s] 

Done in 1.1 min


In [4]:
# Cell 4 — Model definition + load trained weights
# CRITICAL: architecture must be IDENTICAL to the model that produced best_model.pth

def _fps(xyz,n):
    B,N,_=xyz.shape; dev=xyz.device
    c=torch.zeros(B,n,dtype=torch.long,device=dev)
    d=torch.full((B,N),1e10,dtype=torch.float32,device=dev)
    far=torch.randint(0,N,(B,),dtype=torch.long,device=dev)
    bi=torch.arange(B,dtype=torch.long,device=dev)
    for i in range(n):
        c[:,i]=far; cc=xyz[bi,far].unsqueeze(1)
        dd=((xyz-cc)**2).sum(-1)
        d=torch.where(dd<d,dd,d); far=d.argmax(-1)
    return c

def _idx(p,i):
    B=p.shape[0]
    bi=torch.arange(B,device=p.device).view([B]+[1]*(i.dim()-1)).expand_as(i)
    return p[bi,i]

def _ball(nxyz,xyz,r,k):
    d=torch.cdist(nxyz,xyz)
    return torch.where(d<=r,d,d.new_full((),1e10)).topk(k,dim=-1,largest=False).indices

class _MSG(nn.Module):
    def __init__(self,nc,radii,nsamps,ic,specs):
        super().__init__(); self.nc=nc; self.radii=radii; self.nsamps=nsamps
        self.mlps=nn.ModuleList()
        for dims in specs:
            ls,ch=[],ic+3
            for d in dims: ls+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]; ch=d
            self.mlps.append(nn.Sequential(*ls))
    def forward(self,xyz,feat):
        ci=_fps(xyz,self.nc); nxyz=_idx(xyz,ci); outs=[]
        for r,k,mlp in zip(self.radii,self.nsamps,self.mlps):
            idx=_ball(nxyz,xyz,r,k); grp=_idx(feat,idx)
            rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
            grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
            outs.append(mlp(grp).max(2).values)
        return nxyz,torch.cat(outs,1).permute(0,2,1)

class _SA(nn.Module):
    def __init__(self,r,k,ic,dims):
        super().__init__(); self.r=r; self.k=k
        ls,ch=[],ic+3
        for d in dims: ls+=[nn.Conv2d(ch,d,1,bias=False),nn.BatchNorm2d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*ls)
    def forward(self,xyz,feat):
        nc=max(1,xyz.shape[1]//4); ci=_fps(xyz,nc); nxyz=_idx(xyz,ci)
        idx=_ball(nxyz,xyz,self.r,self.k); grp=_idx(feat,idx)
        rxyz=_idx(xyz,idx)-nxyz.unsqueeze(2)
        grp=torch.cat([rxyz,grp],-1).permute(0,3,2,1)
        return nxyz,self.mlp(grp).max(2).values.permute(0,2,1)

class _FP(nn.Module):
    def __init__(self,ic,dims):
        super().__init__()
        ls,ch=[],ic
        for d in dims: ls+=[nn.Conv1d(ch,d,1,bias=False),nn.BatchNorm1d(d),nn.ReLU(True)]; ch=d
        self.mlp=nn.Sequential(*ls)
    def forward(self,xyz1,xyz2,f1,f2):
        d=torch.cdist(xyz1,xyz2); t3=d.topk(3,dim=-1,largest=False)
        w=1./(t3.values+1e-8); w=w/w.sum(-1,keepdim=True)
        interp=(_idx(f2,t3.indices)*w.unsqueeze(-1)).sum(2)
        feat=torch.cat([f1,interp],-1) if f1 is not None else interp
        return self.mlp(feat.permute(0,2,1)).permute(0,2,1)

class DTMPointNet2(nn.Module):
    """MUST match v6 architecture exactly for weight loading to work."""
    def __init__(self,in_feat=9):
        super().__init__()
        C=in_feat
        self.sa1=_MSG(512,[0.5,1.5],[32,64],C,[[64,64],[64,128]])
        self.sa2=_MSG(128,[3.0,8.0],[64,128],192,[[128,192],[128,256]])
        self.sa3=_SA(15.0,128,448,[256,512])
        self.fp3=_FP(512+448,[256,256])
        self.fp2=_FP(256+192,[256,128])
        self.fp1=_FP(128+C,[128,128,64])
        self.head=nn.Sequential(
            nn.Conv1d(64,64,1,bias=False),nn.BatchNorm1d(64),nn.ReLU(True),
            nn.Dropout(0.4),nn.Conv1d(64,2,1))
    def forward(self,pts):
        xyz0=pts[:,:,:3].contiguous(); f0=pts.contiguous()
        xyz1,f1=self.sa1(xyz0,f0); xyz2,f2=self.sa2(xyz1,f1)
        xyz3,f3=self.sa3(xyz2,f2)
        f2=self.fp3(xyz2,xyz3,f2,f3); f1=self.fp2(xyz1,xyz2,f1,f2)
        f0=self.fp1(xyz0,xyz1,f0,f1)
        return self.head(f0.permute(0,2,1)).permute(0,2,1)


IN_FEAT = 9
_base   = DTMPointNet2(IN_FEAT).to(DEVICE)

# ── Load trained weights ──────────────────────────────────────────────────────
# Try SWA first (generally better generalisation), fall back to best_epoch
if SWA_MODEL_PATH and Path(SWA_MODEL_PATH).exists():
    _base.load_state_dict(torch.load(SWA_MODEL_PATH, map_location=DEVICE))
    print(f'Loaded SWA weights: {SWA_MODEL_PATH}')
    LOADED_FROM = 'swa'
else:
    _base.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    print(f'Loaded best_epoch weights: {BEST_MODEL_PATH}')
    LOADED_FROM = 'best_epoch'

model = nn.DataParallel(_base) if N_GPUS > 1 else _base

# Sanity check — make sure architecture matches weights
_x = torch.randn(2, 64, IN_FEAT, device=DEVICE)
with torch.no_grad(): _o = model(_x)
assert _o.shape == (2, 64, 2), f'Shape mismatch: {_o.shape}'
del _x, _o; torch.cuda.empty_cache()

n_p = sum(p.numel() for p in _base.parameters())
print(f'Model: {n_p/1e6:.2f}M params | loaded from: {LOADED_FROM}')
print(f'Starting from 93.65% val accuracy — target: 97%+')


Loaded SWA weights: /kaggle/input/datasets/jaibhagwanchouhan/dtm-v6-final-outputs/swa_model.pth
Model: 0.88M params | loaded from: swa
Starting from 93.65% val accuracy — target: 97%+


In [5]:
# Cell 5 — Dataset with improved Mixup augmentation
# Key changes vs Round 3:
#   1. mixup_alpha from CFG (default 0.2 = sharper mix than 0.4)
#   2. Stronger geometric augmentation during training
#   3. Support for geometric TTA (rotation + scale variants)

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


class GroundDataset(Dataset):
    """
    Round 4 dataset — improved mixup + stronger augmentation.

    Mixup change: Beta(0.2, 0.2) vs old Beta(0.4, 0.4).
    Beta(0.2) peaks sharply at 0 and 1 — most mixed tiles are
    ~90%+ one class, not 50/50. This creates harder boundary cases
    without hallucinating unrealistic blends.

    Augmentation additions:
    - Height jitter ±0.05m (was ±0.02m)
    - Anisotropic XY scale (was only applied sometimes)
    - Point drop 15% (was 10%)
    """
    def __init__(self, cfg, split='train', augment=False,
                 max_tiles=None, pseudo_dir=None, mixup=False,
                 tile_subset=None, tta_angle=None, tta_scale=1.0):
        self.augment    = augment
        self.max_pts    = cfg['max_pts']
        self.seed       = cfg['seed']
        self.split      = split
        self.use_zip    = cfg['use_zip']
        self.mixup      = mixup and split == 'train'
        self.feat_cache = Path(cfg['feat_cache_dir'])/split
        self.pseudo_dir = Path(pseudo_dir) if pseudo_dir else None
        self.cons_dir   = Path(cfg['cons_label_dir'])/split
        self.mixup_alpha= cfg.get('mixup_alpha', 0.2)
        # TTA geometric params (val only)
        self.tta_angle  = tta_angle   # radians, None = no rotation
        self.tta_scale  = tta_scale   # float, 1.0 = no scale

        if self.use_zip:
            self._zpath = cfg['zip_path']
            self._tiles = self._disc()
        else:
            self._td = Path(cfg['training_dir'])/split
            self._tiles = sorted(
                p.name for p in self._td.glob('tile_*')
                if (p/'labels.npy').exists())

        if tile_subset is not None:
            self._tiles = [t for t in self._tiles if t in set(tile_subset)]

        if max_tiles and len(self._tiles) > max_tiles:
            rng  = np.random.default_rng(self.seed)
            pick = rng.choice(len(self._tiles), max_tiles, replace=False)
            self._tiles = [self._tiles[i] for i in sorted(pick)]

        src = 'pseudo' if pseudo_dir else 'consensus'
        print(f'  [{split}] {len(self._tiles)} tiles '
              f'| lbl={src} | mixup={self.mixup}')

    def _disc(self):
        with zipfile.ZipFile(self._zpath) as zf: names=zf.namelist()
        t=set()
        for n in names:
            p=PurePosixPath(n).parts
            if self.split in p and 'tile_' in p[-2]:
                t.add(p[p.index(self.split)+1])
        return sorted(t)

    def _load_zip(self,tile,fn):
        if not hasattr(self,'_zf') or self._zf is None:
            self._zf = zipfile.ZipFile(self._zpath)
        with self._zf.open(f'{self.split}/{tile}/{fn}') as fh:
            return np.load(io.BytesIO(fh.read()))

    def _get_labels(self,tile):
        if self.pseudo_dir is not None:
            p = self.pseudo_dir/f'{tile}.npy'
            if p.exists(): return np.load(str(p))
        p = self.cons_dir/f'{tile}.npy'
        if p.exists(): return np.load(str(p))
        if self.use_zip: return self._load_zip(tile,'labels.npy')
        return np.load(str(self._td/tile/'labels.npy'))

    def _load_tile(self, idx):
        tile = self._tiles[idx]
        rng  = np.random.default_rng(idx + self.seed*10000)
        xyz  = (self._load_zip(tile,'points.npy') if self.use_zip
                else np.load(str(self._td/tile/'points.npy')))
        lbl  = self._get_labels(tile).astype(np.float32)

        # Apply geometric TTA if set (val only, deterministic)
        if self.tta_angle is not None:
            t = self.tta_angle
            R = np.array([[np.cos(t),-np.sin(t)],[np.sin(t),np.cos(t)]],np.float32)
            xyz[:,:2] = xyz[:,:2] @ R.T
            xyz *= self.tta_scale

        if self.augment:
            t = rng.uniform(0, 2*np.pi)
            R = np.array([[np.cos(t),-np.sin(t)],[np.sin(t),np.cos(t)]],np.float32)
            xyz[:,:2] = xyz[:,:2]@R.T
            # Stronger height jitter (±0.05 vs ±0.02)
            xyz[:,2] += rng.normal(0,0.05,len(xyz)).astype(np.float32)
            xyz *= rng.uniform(0.90,1.10)
            if rng.random()>.5: xyz[:,0]*=-1
            if rng.random()>.5: xyz[:,1]*=-1
            # Anisotropic XY scale (always, not just sometimes)
            xyz[:,0] *= rng.uniform(0.85,1.15)
            xyz[:,1] *= rng.uniform(0.85,1.15)
            xyz[:,2] *= rng.uniform(0.85,1.15)
            # Stronger point drop (15% vs 10%)
            keep = rng.random(len(xyz)) > 0.15
            if keep.sum() > 64: xyz=xyz[keep]; lbl=lbl[keep]

        N = len(xyz)
        if N > self.max_pts:
            ch=rng.choice(N,self.max_pts,replace=False)
            xyz=xyz[ch]; lbl=lbl[ch]
        elif N < self.max_pts:
            pad=rng.choice(N,self.max_pts-N,replace=True)
            xyz=np.concatenate([xyz,xyz[pad]])
            lbl=np.concatenate([lbl,lbl[pad]])

        cp = self.feat_cache/f'{tile}.npy'
        # Recompute features when augmenting (geometry changed)
        f6 = (np.load(str(cp)) if cp.exists() and not self.augment
                                                and self.tta_angle is None
              else geo_features(xyz))
        if len(f6) != self.max_pts: f6 = geo_features(xyz)

        xn=xyz.copy(); xn[:,:2]-=xn[:,:2].mean(0)
        xn[:,2]-=float(np.median(xn[:,2])); xn/=(np.abs(xn).max()+1e-6)
        return np.concatenate([xn,f6],axis=1).astype(np.float32), lbl

    def __len__(self): return len(self._tiles)

    def __getitem__(self, idx):
        pts_a, lbl_a = self._load_tile(idx)

        if self.mixup and np.random.random() < 0.5:
            idx_b = np.random.randint(0, len(self._tiles))
            pts_b, lbl_b = self._load_tile(idx_b)
            # Beta(mixup_alpha, mixup_alpha) — sharper than old 0.4
            lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))
            pts_a = lam * pts_a + (1 - lam) * pts_b
            lbl_a = lam * lbl_a + (1 - lam) * lbl_b

        lbl_hard = (lbl_a >= 0.5).astype(np.int64)
        return torch.from_numpy(pts_a), torch.from_numpy(lbl_hard)


def make_loader(cfg, split, augment=False, pseudo_dir=None,
                balanced=False, mixup=False, tile_subset=None,
                batch_size=None, tta_angle=None, tta_scale=1.0):
    ds = GroundDataset(cfg, split, augment=augment,
                       pseudo_dir=pseudo_dir, mixup=mixup,
                       tile_subset=tile_subset,
                       tta_angle=tta_angle, tta_scale=tta_scale)
    bs = batch_size or cfg['batch_size']
    nw = cfg['num_workers']
    kw = dict(num_workers=nw, pin_memory=True,
              persistent_workers=(nw>0),
              prefetch_factor=cfg['prefetch_factor'] if nw>0 else None)

    if balanced and split == 'train':
        ws = []
        for tile in ds._tiles:
            lbl = ds._get_labels(tile).astype(float)
            gf  = lbl.mean(); nf = 1-gf
            ws.append(1./(min(gf,nf)+0.05))
        sampler = WeightedRandomSampler(ws, len(ws), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler,
                          drop_last=True, **kw)

    shuffle = (split == 'train')
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      drop_last=shuffle, **kw)


# Smoke test
_ds = GroundDataset(CFG, 'train', max_tiles=2)
_p, _l = _ds[0]
assert _p.shape == (CFG['max_pts'], 9)
del _ds, _p, _l
print('Dataset smoke test PASSED')


  [train] 2 tiles | lbl=consensus | mixup=False
Dataset smoke test PASSED


In [6]:
# Cell 6 — Round 4: Generate pseudo-labels (conf >= 0.92, wider net than R3)
#
# KEY CHANGE vs Round 3:
#   R3 used conf=0.95 → replaced only 2.7% of labels (very conservative)
#   R4 uses conf=0.92 → will replace ~8-12% of labels
#   This casts a wider net, capturing more correctable uncertain labels.
#
# We also add a helper to REFRESH pseudo-labels during Phase A training.
# This is the core fix for the "same tiles loop" problem.

def build_consensus_cache(cfg, split):
    """Build consensus labels if not already present."""
    out_dir = Path(cfg['cons_label_dir'])/split
    out_dir.mkdir(parents=True, exist_ok=True)

    SCALES = [(1.0,0.20),(2.0,0.35),(4.0,0.60),(8.0,1.00)]

    def _consensus(xyz):
        z=xyz[:,2].astype(np.float32); votes=np.zeros(len(xyz),np.int32)
        for cm,th in SCALES:
            xmin,ymin=xyz[:,0].min(),xyz[:,1].min()
            xr=xyz[:,0].max()-xmin+1e-6; yr=xyz[:,1].max()-ymin+1e-6
            GW=int(np.clip(xr/cm,8,64)); GH=int(np.clip(yr/cm,8,64))
            xi=np.clip(((xyz[:,0]-xmin)/xr*GW).astype(np.int32),0,GW-1)
            yi=np.clip(((xyz[:,1]-ymin)/yr*GH).astype(np.int32),0,GH-1)
            ci=yi*GW+xi; c_min=np.full(GW*GH,np.inf,np.float32)
            np.minimum.at(c_min,ci,z); c_min[np.isinf(c_min)]=z.mean()
            votes+=(z-c_min[ci]<=th).astype(np.int32)
        return (votes>=3).astype(np.int8)

    if cfg['use_zip']:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tiles={}
        for n in names:
            parts=PurePosixPath(n).parts
            if split in parts and n.endswith('points.npy'):
                tiles[parts[parts.index(split)+1]]=n
        zf_h=zipfile.ZipFile(cfg['zip_path'])
        try:
            for t,zp in tqdm(tiles.items(),desc=f'Consensus[{split}]',unit='tile'):
                out=out_dir/f'{t}.npy'
                if out.exists(): continue
                with zf_h.open(zp) as fh: xyz=np.load(io.BytesIO(fh.read()))
                np.save(str(out),_consensus(xyz))
        finally: zf_h.close()
    else:
        for td in tqdm(sorted((Path(cfg['training_dir'])/split).glob('tile_*')),
                       desc=f'Consensus[{split}]'):
            out=out_dir/f'{td.name}.npy'
            if not out.exists():
                np.save(str(out),_consensus(np.load(str(td/'points.npy'))))


n_cons_train = len(list((Path(CFG['cons_label_dir'])/'train').glob('*.npy')))
n_cons_val   = len(list((Path(CFG['cons_label_dir'])/'val').glob('*.npy')))
print(f'Consensus labels: train={n_cons_train}, val={n_cons_val}')
if n_cons_train == 0:
    print('Building consensus labels...')
    build_consensus_cache(CFG,'train'); build_consensus_cache(CFG,'val')
else:
    print('Consensus labels exist — skipping rebuild')


def generate_pseudolabels(model, cfg, conf_thresh, round_name, force=False):
    """
    Generate pseudo-labels for training tiles.
    
    NEW in Round 4:
    - force=True wipes and regenerates (for mid-training refreshes)
    - conf_thresh can be lower (0.88-0.92) for wider coverage
    - Uses multi-subsample averaging for more stable probabilities
    """
    out_dir = Path(cfg['pseudo_dir'])/round_name
    if force and out_dir.exists():
        import shutil
        shutil.rmtree(str(out_dir))
        print(f'  Wiped stale pseudo-labels: {out_dir}')
    out_dir.mkdir(parents=True, exist_ok=True)

    base = model.module if hasattr(model,'module') else model
    base.eval()

    use_zip = cfg['use_zip']
    if use_zip:
        with zipfile.ZipFile(cfg['zip_path']) as zf: names=zf.namelist()
        tile_map={}
        for n in names:
            parts=PurePosixPath(n).parts
            if 'train' in parts and n.endswith('points.npy'):
                tile_map[parts[parts.index('train')+1]]=n
        tile_list=sorted(tile_map.items())
    else:
        tdir=Path(cfg['training_dir'])/'train'
        tile_list=[(p.name,str(p/'points.npy')) for p in sorted(tdir.glob('tile_*'))]

    cons_dir=Path(cfg['cons_label_dir'])/'train'
    rng=np.random.default_rng(cfg['seed'])
    max_pts=cfg['max_pts']
    n_repl=0; n_total=0

    # Multi-subsample averaging: 3 passes per tile → more stable probs
    N_SUBSAMPLE = 3

    zf_h = zipfile.ZipFile(cfg['zip_path']) if use_zip else None
    try:
        for ti,(tile,pts_path) in enumerate(tqdm(
                tile_list,desc=f'Pseudo-labels [{round_name}] (conf>={conf_thresh})',
                unit='tile')):
            out_file = out_dir/f'{tile}.npy'
            if out_file.exists(): n_total+=1; continue

            if use_zip:
                with zf_h.open(pts_path) as fh: xyz=np.load(io.BytesIO(fh.read()))
            else: xyz=np.load(pts_path)
            N=len(xyz); n_total+=N

            # Multi-subsample: run model N_SUBSAMPLE times, average probs
            avg_probs = np.zeros(N, dtype=np.float32)
            for _pass in range(N_SUBSAMPLE):
                rng_pass = np.random.default_rng(cfg['seed'] + ti*100 + _pass)
                if N > max_pts:
                    sub_idx = rng_pass.choice(N, max_pts, replace=False)
                    sub = xyz[sub_idx]
                else:
                    sub_idx = np.arange(N); sub = xyz

                f6 = geo_features(sub)
                xn = sub.copy(); xn[:,:2] -= xn[:,:2].mean(0)
                xn[:,2] -= float(np.median(xn[:,2])); xn /= (np.abs(xn).max()+1e-6)
                pts9 = np.concatenate([xn,f6],axis=1).astype(np.float32)

                tensor = torch.from_numpy(pts9).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    with torch.amp.autocast('cuda'):
                        logits = base(tensor)
                    probs = F.softmax(logits[0],-1)[:,1].cpu().numpy()
                del tensor, logits

                if N > max_pts:
                    from scipy.spatial import cKDTree
                    _,ni = cKDTree(sub).query(xyz, k=1, workers=-1)
                    avg_probs += probs[ni].astype(np.float32)
                else:
                    avg_probs += probs.astype(np.float32)
                del probs, sub, f6, xn, pts9

            avg_probs /= N_SUBSAMPLE
            torch.cuda.empty_cache()

            conf = np.abs(avg_probs - 0.5) * 2.
            model_lbl = (avg_probs >= 0.5).astype(np.int8)

            cp = cons_dir/f'{tile}.npy'
            baseline = np.load(str(cp)).astype(np.int8) if cp.exists() else model_lbl.copy()
            refined = baseline.copy()
            hc = conf >= conf_thresh
            refined[hc] = model_lbl[hc]; n_repl += int(hc.sum())
            del xyz, conf, model_lbl, baseline, hc, avg_probs
            np.save(str(out_file), refined.astype(np.int8))
            del refined
            if ti%500==0 and ti>0: gc.collect()
    finally:
        if zf_h: zf_h.close()

    gc.collect(); torch.cuda.empty_cache()
    pct = 100*n_repl/max(n_total,1)
    print(f'  Replaced {pct:.1f}% of labels (conf>={conf_thresh})')
    print(f'  (multi-subsample averaging: {N_SUBSAMPLE} passes/tile)')
    return str(out_dir)


print('Generating Round 4 pseudo-labels (conf=0.92, wider than R3=0.95)...')
PSEUDO_DIR_R4 = generate_pseudolabels(
    model, CFG, conf_thresh=CFG['pseudo_conf_r4'],
    round_name='Round4', force=False)
print(f'Pseudo-labels saved: {PSEUDO_DIR_R4}')

# This is the active pseudo dir — updated during Phase A refreshes
ACTIVE_PSEUDO_DIR = PSEUDO_DIR_R4


Consensus labels: train=0, val=0
Building consensus labels...


Consensus[val]: 100%|██████████| 1216/1216 [00:02<00:00, 431.83it/s]


Generating Round 4 pseudo-labels (conf=0.92, wider than R3=0.95)...


Pseudo-labels [Round4] (conf>=0.92): 100%|██████████| 4922/4922 [25:39<00:00,  3.20tile/s]

  Replaced 11.6% of labels (conf>=0.92)
  (multi-subsample averaging: 3 passes/tile)
Pseudo-labels saved: /kaggle/working/pseudo_labels/Round4


In [7]:
# Cell 7 — Score tiles for Hard Example Mining (initial scoring)
#
# KEY CHANGE vs Round 3:
# - HEM is now run 3 INDEPENDENT ROUNDS in Phase B (not one fixed set)
# - After each HEM round, we RE-MINE tiles (model has improved)
# - This prevents the "training same tiles 12 times" problem
# - This function is called once here, then re-called between HEM rounds

def score_tiles_for_hem(model, cfg, pseudo_dir, desc='Scoring tiles'):
    """
    Score all training tiles. Returns sorted list of (accuracy, tile_name).
    Called before each HEM round so the hard set is always fresh.
    """
    base = model.module if hasattr(model,'module') else model
    base.eval()

    score_ds = GroundDataset(cfg, 'train', augment=False, pseudo_dir=pseudo_dir)
    score_ld = DataLoader(score_ds, batch_size=cfg['batch_size'],
                           shuffle=False, drop_last=False,
                           num_workers=cfg['num_workers'],
                           pin_memory=True)

    tile_accs = []
    tile_idx  = 0

    with torch.no_grad():
        for pts, lbs in tqdm(score_ld, desc=desc, ncols=88):
            pts = pts.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                logits = base(pts)
            preds = logits.argmax(-1)

            for b in range(pts.shape[0]):
                if tile_idx >= len(score_ds._tiles): break
                t_acc = (preds[b].cpu() == lbs[b]).float().mean().item()
                tile_accs.append((t_acc, score_ds._tiles[tile_idx]))
                tile_idx += 1

    torch.cuda.empty_cache(); gc.collect()
    tile_accs.sort(key=lambda x: x[0])  # worst first

    n_hard = int(len(tile_accs) * cfg['hem_hard_frac'])
    hard   = [t for _, t in tile_accs[:n_hard]]
    medium = [t for _, t in tile_accs[n_hard:]]

    print(f'  Total tiles     : {len(tile_accs)}')
    print(f'  Hard (worst {cfg["hem_hard_frac"]*100:.0f}%): {n_hard}  '
          f'mean_acc={np.mean([a for a,_ in tile_accs[:n_hard]])*100:.2f}%  '
          f'min_acc={tile_accs[0][0]*100:.2f}%')
    print(f'  Easy (best {(1-cfg["hem_hard_frac"])*100:.0f}%): {len(medium)}  '
          f'mean_acc={np.mean([a for a,_ in tile_accs[n_hard:]])*100:.2f}%')
    print(f'  Worst 5 tiles:')
    for acc,tile in tile_accs[:5]:
        print(f'    {tile}  acc={acc*100:.1f}%')

    return hard, medium, tile_accs


print('Scoring all training tiles for initial HEM...')
print('(~5 min — runs model on every tile once)')
HARD_TILES, MEDIUM_TILES, ALL_TILE_ACCS = score_tiles_for_hem(
    model, CFG, ACTIVE_PSEUDO_DIR, desc='Initial tile scoring')


Scoring all training tiles for initial HEM...
(~5 min — runs model on every tile once)
  [train] 4922 tiles | lbl=pseudo | mixup=False


Initial tile scoring: 100%|███████████████████████████| 308/308 [00:55<00:00,  5.55it/s]

  Total tiles     : 4922
  Hard (worst 35%): 1722  mean_acc=79.60%  min_acc=47.07%
  Easy (best 65%): 3200  mean_acc=90.23%
  Worst 5 tiles:
    tile_000761  acc=47.1%
    tile_004011  acc=50.2%
    tile_004332  acc=54.7%
    tile_005776  acc=57.0%
    tile_005210  acc=58.0%


In [8]:
# Cell 8 — Fine-tuning: Phase A (refresh labels) + Phase B (multi-round HEM) + Phase C (SWA)
#
# ROUND 4 KEY CHANGES:
#
# Phase A — Full-dataset fine-tune WITH pseudo-label refresh every 5 epochs
#   Old: static labels all 15 epochs → model memorises same pseudo-labels
#   New: refresh labels every 5 epochs → model trains on improving labels
#
# Phase B — 3 INDEPENDENT HEM ROUNDS (not 12 epochs on same tiles)
#   Old: score once → train on same 40% tiles for 12 epochs
#   New: score → train 8 ep → re-score → train 8 ep → re-score → train 8 ep
#   Each round the model is stronger, so re-mining finds NEW hard tiles
#
# Phase C — SWA with 8 epochs + 1200-tile BN update (was 5ep + 400 tiles)
#
# Expected: ~96-98%

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.optim.swa_utils import AveragedModel, SWALR
import contextlib


class FocalLoss(nn.Module):
    def __init__(self,a,g,s):
        super().__init__(); self.a=a; self.g=g; self.s=s
    def forward(self,logits,targets):
        nc=logits.size(-1)
        soft=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1.)
        soft=soft*(1-self.s)+self.s/nc
        ce=-(soft*torch.log_softmax(logits,-1)).sum(-1)
        pt=torch.exp(-ce)
        at=torch.where(targets==1,
            targets.new_full(targets.shape,self.a,dtype=torch.float32),
            targets.new_full(targets.shape,1-self.a,dtype=torch.float32))
        return (at*(1-pt)**self.g*ce).mean()


def _metrics(preds,labels):
    acc=(preds==labels).float().mean().item()
    tp=((preds==1)&(labels==1)).sum().item()
    fp=((preds==1)&(labels==0)).sum().item()
    fn=((preds==0)&(labels==1)).sum().item()
    p=tp/(tp+fp+1e-9); r=tp/(tp+fn+1e-9)
    return acc,2*p*r/(p+r+1e-9)


def _autocast():
    return torch.amp.autocast('cuda', enabled=CFG['use_amp'])

def _make_scaler():
    return torch.amp.GradScaler('cuda', enabled=CFG['use_amp'])


def run_epoch(model, loader, opt, scaler, crit, base,
              train=True, desc=''):
    model.train() if train else model.eval()
    total_loss=0.; all_p=[]; all_l=[]

    if train:
        for pts,lbs in tqdm(loader,desc=desc,leave=False,ncols=90):
            pts=pts.to(DEVICE,non_blocking=True)
            lbs=lbs.to(DEVICE,non_blocking=True)
            opt.zero_grad()
            with _autocast():
                logits=model(pts)
                loss=crit(logits.view(-1,2),lbs.view(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(base.parameters(),CFG['grad_clip'])
            scaler.step(opt); scaler.update()
            total_loss+=loss.item()
            with torch.no_grad():
                all_p.append(logits.detach().argmax(-1).view(-1).cpu())
                all_l.append(lbs.view(-1).cpu())
    else:
        with torch.no_grad():
            for pts,lbs in tqdm(loader,desc=desc,leave=False,ncols=90):
                pts=pts.to(DEVICE,non_blocking=True)
                lbs=lbs.to(DEVICE,non_blocking=True)
                with _autocast():
                    logits=model(pts)
                    loss=crit(logits.view(-1,2),lbs.view(-1))
                total_loss+=loss.item()
                all_p.append(logits.detach().argmax(-1).view(-1).cpu())
                all_l.append(lbs.view(-1).cpu())

    acc,f1=_metrics(torch.cat(all_p),torch.cat(all_l))
    return total_loss/len(loader),acc,f1


def val_epoch(model, cfg, crit, base, pseudo_dir, desc='val'):
    """Standard val pass (no TTA — fast, used during training)."""
    va_ld = make_loader(cfg,'val',augment=False,
                        pseudo_dir=None)  # val uses consensus labels
    _,va,vf1 = run_epoch(model,va_ld,None,None,crit,base,
                          train=False,desc=desc)
    return va, vf1


def budget_ok():
    elapsed_min = (time.time() - SESSION_START) / 60
    return elapsed_min < 240  # 240 min hard cap (10 min safety buffer)


crit   = FocalLoss(CFG['focal_alpha'],CFG['focal_gamma'],CFG['label_smooth'])
base   = model.module if hasattr(model,'module') else model
ALL_H  = []; best_acc = 0.0

Path(CFG['logs_dir']).mkdir(parents=True,exist_ok=True)

# current pseudo dir — updated on refresh
cur_pseudo = ACTIVE_PSEUDO_DIR


# ══════════════════════════════════════════════════════════════════════════════
# SUB-PHASE A: Full-dataset fine-tune + periodic pseudo-label refresh
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  SUB-PHASE A: Fine-tune + label refresh every '
      f'{CFG["phaseA_refresh_every"]} epochs')
print(f'  Epochs: {CFG["phaseA_epochs"]}  LR: {CFG["phaseA_lr"]}')
print('='*60)

opt_A   = torch.optim.AdamW(base.parameters(),
                              lr=CFG['phaseA_lr'],
                              weight_decay=CFG['weight_decay'])
sched_A = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_A, T_max=CFG['phaseA_epochs'], eta_min=CFG['phaseA_lr']*0.01)
scaler  = _make_scaler()

for ep in range(1, CFG['phaseA_epochs']+1):
    if not budget_ok(): print('Budget reached (Phase A)'); break

    # ── Pseudo-label refresh ────────────────────────────────────────────────
    # OOM note: pseudo-label generation runs model inference while the
    # optimizer + scaler + train loader are all live. We free the train
    # loader and clear cache first, then rebuild it after.
    if ep > 1 and (ep-1) % CFG['phaseA_refresh_every'] == 0:
        print(f'\n  [Ep {ep}] Refreshing pseudo-labels (conf={CFG["pseudo_conf_refresh"]})...')
        # Free the train loader to release prefetch buffers before inference
        del tr_ld_A; gc.collect(); torch.cuda.empty_cache()
        refresh_name = f'Round4_refresh_ep{ep}'
        cur_pseudo = generate_pseudolabels(
            model, CFG,
            conf_thresh=CFG['pseudo_conf_refresh'],
            round_name=refresh_name,
            force=True)
        print(f'  Labels refreshed → {cur_pseudo}')
        # Loader will be rebuilt at top of next iteration

    # Rebuild loader with current pseudo-labels
    tr_ld_A = make_loader(CFG,'train',augment=True,pseudo_dir=cur_pseudo,
                           balanced=True,mixup=True)

    tl,ta,tf1 = run_epoch(model,tr_ld_A,opt_A,scaler,crit,base,
                           train=True,desc=f'A Ep{ep:2d} train')
    sched_A.step()
    cur_lr = opt_A.param_groups[0]['lr']

    va,vf1 = float('nan'),float('nan')
    if ep % CFG['val_every'] == 0 or ep == CFG['phaseA_epochs']:
        va,vf1 = val_epoch(model,CFG,crit,base,cur_pseudo,desc=f'A Ep{ep:2d} val')
        if va > best_acc:
            best_acc = va
            torch.save(base.state_dict(), CFG['model_save'])

    star = ' BEST' if (not math.isnan(va) and va == best_acc) else ''
    print(f'A Ep{ep:2d} lr={cur_lr:.5f} tl={tl:.4f} ta={ta:.4f} '
          f'| va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')
    ALL_H.append(dict(phase='A',ep=ep,tl=tl,ta=ta,va=va,vf1=vf1))
    torch.save(dict(epoch=ep,phase='A',model=base.state_dict(),
                    best_acc=best_acc),CFG['ckpt'])

print(f'\nPhase A complete. Best: {best_acc*100:.2f}%')


# ══════════════════════════════════════════════════════════════════════════════
# SUB-PHASE B: Multi-round HEM (re-mine tiles after each round)
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print(f'  SUB-PHASE B: {CFG["phaseB_rounds"]} HEM Rounds (re-mine after each)')
print(f'  Epochs per round: {CFG["phaseB_epochs_per_round"]}  '
      f'LR: {CFG["phaseB_lr"]}  Hard frac: {CFG["hem_hard_frac"]}')
print('='*60)

for hem_round in range(1, CFG['phaseB_rounds']+1):
    if not budget_ok(): print('Budget reached (Phase B)'); break

    print(f'\n  ── HEM Round {hem_round}/{CFG["phaseB_rounds"]} ──')

    # Always reload best weights at the start of each HEM round
    base.load_state_dict(torch.load(CFG['model_save'], map_location=DEVICE))

    # Re-score tiles with current model (finds NEW hard tiles each round)
    # OOM note: inference runs while optimizer from previous HEM round is
    # still in scope. Explicitly delete the previous loader + optimizer
    # before scoring to free prefetch buffers and optimizer state tensors.
    print(f'  Re-mining tiles (round {hem_round})...')
    if hem_round > 1:
        try: del tr_ld_B
        except NameError: pass
        try: del opt_B, sched_B
        except NameError: pass
        gc.collect(); torch.cuda.empty_cache()
    HARD_TILES, _, _ = score_tiles_for_hem(
        model, CFG, cur_pseudo,
        desc=f'HEM-R{hem_round} scoring')

    tr_ld_B = make_loader(CFG,'train',augment=True,pseudo_dir=cur_pseudo,
                           balanced=True,mixup=False,tile_subset=HARD_TILES)

    opt_B   = torch.optim.AdamW(base.parameters(),
                                  lr=CFG['phaseB_lr'],
                                  weight_decay=CFG['weight_decay'])
    sched_B = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_B, T_max=CFG['phaseB_epochs_per_round'],
        eta_min=CFG['phaseB_lr']*0.05)

    for ep in range(1, CFG['phaseB_epochs_per_round']+1):
        if not budget_ok(): print('Budget reached mid-HEM'); break

        tl,ta,tf1 = run_epoch(model,tr_ld_B,opt_B,scaler,crit,base,
                               train=True,desc=f'B-R{hem_round} Ep{ep:2d} train')
        sched_B.step()
        cur_lr = opt_B.param_groups[0]['lr']

        va,vf1 = float('nan'),float('nan')
        if ep % CFG['val_every'] == 0 or ep == CFG['phaseB_epochs_per_round']:
            va,vf1 = val_epoch(model,CFG,crit,base,cur_pseudo,
                               desc=f'B-R{hem_round} Ep{ep:2d} val')
            if va > best_acc:
                best_acc = va
                torch.save(base.state_dict(), CFG['model_save'])

        star = ' BEST' if (not math.isnan(va) and va == best_acc) else ''
        print(f'B-R{hem_round} Ep{ep:2d} lr={cur_lr:.5f} tl={tl:.4f} ta={ta:.4f} '
              f'| va={va:.4f} vf1={vf1:.4f} best={best_acc:.4f}{star}')
        ALL_H.append(dict(phase=f'B_R{hem_round}',ep=ep,tl=tl,ta=ta,va=va,vf1=vf1))

print(f'\nPhase B complete. Best: {best_acc*100:.2f}%')


# ══════════════════════════════════════════════════════════════════════════════
# SUB-PHASE C: SWA — OOM-safe, 3-stage teardown
# ══════════════════════════════════════════════════════════════════════════════
# OOM anatomy (why it always dies here on T4 x2):
#   GPU holds simultaneously:
#     - training model weights + optimizer states  (~48 MB)
#     - AveragedModel = full second weight copy    (~16 MB)
#     - update_bn DataLoader activations, both GPUs via DataParallel  (~doubled)
#   Total spikes well past 15.6 GB → silent kernel death
#
# Fix: 3-stage teardown before BN update
#   Stage 1 — extract averaged weights to CPU dict (no model object on GPU)
#   Stage 2 — destroy EVERYTHING on GPU: model, DataParallel, optimizer, scaler, loader
#   Stage 3 — build a brand-new CPU-only model, load averaged weights, run BN on CPU
#
# This keeps GPU memory at ~0 during BN update. Safe for T4 x2.

print('\n' + '='*60)
print(f'  SUB-PHASE C: SWA ({CFG["swa_epochs"]} epochs) — OOM-safe teardown')
print('='*60)

# Reload best weights fresh before SWA accumulation
base.load_state_dict(torch.load(CFG['model_save'], map_location=DEVICE))

# AveragedModel lives on GPU alongside training model — that's fine during training
swa_m  = AveragedModel(base)
opt_C  = torch.optim.AdamW(base.parameters(), lr=CFG['swa_lr'],
                             weight_decay=CFG['weight_decay'])
swa_s  = SWALR(opt_C, swa_lr=CFG['swa_lr'])
tr_ld_C = make_loader(CFG,'train',augment=True,pseudo_dir=cur_pseudo,
                       balanced=True,mixup=False)

for ep in range(1, CFG['swa_epochs']+1):
    if not budget_ok(): print('Budget reached (Phase C)'); break
    tl,ta,tf1 = run_epoch(model,tr_ld_C,opt_C,scaler,crit,base,
                           train=True,desc=f'C Ep{ep} swa')
    swa_m.update_parameters(base); swa_s.step()
    print(f'C Ep{ep} tl={tl:.4f} ta={ta:.4f} (SWA accumulating)')

# ── STAGE 1: Extract averaged weights to a plain CPU state dict ───────────────
# We pull only the numbers, not the model object. This is the key step —
# swa_m.module holds the EMA of all parameters; .cpu().state_dict() is a
# plain dict of tensors. We never run inference through swa_m again.
print('SWA Stage 1: extracting averaged weights to CPU dict...')
swa_state_dict = {k: v.cpu().clone() for k, v in swa_m.module.state_dict().items()}

# ── STAGE 2: Destroy EVERYTHING on GPU ───────────────────────────────────────
# Order matters: loader → scaler → optimizer → swa_m → model → base → sync
print('SWA Stage 2: full GPU teardown...')
del tr_ld_C
del swa_s, opt_C
del swa_m          # second weight copy gone
del scaler
# Unwrap DataParallel if present, then delete
if hasattr(model, 'module'):
    del model.module
del model, base
torch.cuda.empty_cache()
gc.collect()
torch.cuda.synchronize()
# Verify GPU is clear
for i in range(N_GPUS):
    used = torch.cuda.memory_allocated(i) / 1e6
    print(f'  GPU {i} after teardown: {used:.1f} MB allocated (target: <100 MB)')

# ── STAGE 3: CPU-only model for BN update ────────────────────────────────────
# Build a fresh CPU model, load the averaged weights, run update_bn on CPU.
# No GPU involved. No OOM possible.
print(f'SWA Stage 3: BN update on CPU ({CFG["swa_bn_tiles"]} tiles)...')
swa_cpu = DTMPointNet2(IN_FEAT)   # CPU, no .to(DEVICE)
swa_cpu.load_state_dict(swa_state_dict)
del swa_state_dict

import random as _rand
_bn_ds = GroundDataset(CFG,'train',augment=False,pseudo_dir=cur_pseudo)
_bn_tiles = _rand.sample(_bn_ds._tiles, min(CFG['swa_bn_tiles'], len(_bn_ds._tiles)))
_bn_ds._tiles = _bn_tiles
# batch_size=8 on CPU is fine — no GPU activation footprint
_bn_ld = DataLoader(_bn_ds, batch_size=8, shuffle=True,
                     num_workers=0, pin_memory=False)

torch.optim.swa_utils.update_bn(_bn_ld, swa_cpu, device=torch.device('cpu'))
torch.save(swa_cpu.state_dict(), CFG['swa_save'])
print(f'SWA saved: {CFG["swa_save"]}  ({Path(CFG["swa_save"]).stat().st_size/1e3:.0f} KB)')
del swa_cpu, _bn_ld, _bn_ds; gc.collect()

# ── Rebuild model on GPU for Cell 9 TTA sweep ────────────────────────────────
# Cell 9 needs 'base' and 'model' to exist; rebuild them from best_model.pth.
# This is cheap (just weight loading, no training) and safe.
print('Rebuilding GPU model for TTA sweep (Cell 9)...')
_base_new = DTMPointNet2(IN_FEAT).to(DEVICE)
_base_new.load_state_dict(torch.load(CFG['model_save'], map_location=DEVICE))
model = nn.DataParallel(_base_new) if N_GPUS > 1 else _base_new
base  = _base_new
del _base_new
print('GPU model ready.')

# Save training history
with open(CFG['history'],'w') as f: json.dump(ALL_H,f,indent=2)

print(f'\nFine-tuning complete. Best val accuracy: {best_acc*100:.2f}%')
print(f'Active pseudo-labels: {cur_pseudo}')

# Store cur_pseudo for Cell 9
FINAL_PSEUDO_DIR = cur_pseudo



  SUB-PHASE A: Fine-tune + label refresh every 5 epochs
  Epochs: 20  LR: 0.0002
  [train] 4922 tiles | lbl=pseudo | mixup=True


A Ep 1 lr=0.00020 tl=0.0226 ta=0.8730 | va=nan vf1=nan best=0.0000
  [train] 4922 tiles | lbl=pseudo | mixup=True


  [val] 1216 tiles | lbl=consensus | mixup=False


A Ep 2 lr=0.00020 tl=0.0195 ta=0.8767 | va=0.9077 vf1=0.8723 best=0.9077 BEST
  [train] 4922 tiles | lbl=pseudo | mixup=True


A Ep 3 lr=0.00019 tl=0.0193 ta=0.8780 | va=nan vf1=nan best=0.9077
  [train] 4922 tiles | lbl=pseudo | mixup=True


...

C Ep1 tl=0.0154 ta=0.9065 (SWA accumulating)


C Ep2 tl=0.0153 ta=0.9077 (SWA accumulating)


C Ep3 tl=0.0151 ta=0.9088 (SWA accumulating)


C Ep4 tl=0.0151 ta=0.9090 (SWA accumulating)


C Ep5 tl=0.0152 ta=0.9082 (SWA accumulating)
Budget reached (Phase C)
SWA Stage 1: extracting averaged weights to CPU dict...
SWA Stage 2: full GPU teardown...
  GPU 0 after teardown: 38.2 MB allocated (target: <100 MB)
  GPU 1 after teardown: 8.5 MB allocated (target: <100 MB)
SWA Stage 3: BN update on CPU (1200 tiles)...
  [train] 4922 tiles | lbl=pseudo | mixup=False
SWA saved: /kaggle/working/logs/swa_model.pth  (3575 KB)
Rebuilding GPU model for TTA sweep (Cell 9)...
GPU model ready.

Fine-tuning complete. Best val accuracy: 91.68%
Active pseudo-labels: /kaggle/working/pseudo_labels/Round4_refresh_ep16


In [ ]:
# Cell 9 — Geometric TTA Threshold Sweep + Package
#
# KEY CHANGE vs Round 3:
#   Old TTA: 8 random permutation passes (just subsamples points)
#   New TTA: 12 GEOMETRIC variants = 4 rotations × 3 scales
#     Rotations: 0°, 90°, 180°, 270°
#     Scales: 0.95, 1.00, 1.05
#   This covers the actual geometric space the model may have seen,
#   not just random reorderings of the same tile.
#
# Additionally we sweep BOTH models (best_epoch, SWA) and pick winner.
# Expected: +0.5-0.8pp vs permutation-only TTA

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def sweep_geometric_tta(model_path, cfg, device, label='', n_rotations=4, n_scales=3):
    """
    Geometric TTA threshold sweep.
    
    TTA variants:
      - n_rotations: evenly spaced rotations (0, 90, 180, 270 deg)
      - n_scales: scale factors around 1.0 (e.g., 0.95, 1.00, 1.05)
    Total passes = n_rotations * n_scales
    """
    if not Path(model_path).exists():
        print(f'  [SKIP] {model_path} not found')
        return None

    m = DTMPointNet2(IN_FEAT).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()

    rotations = [2*np.pi*i/n_rotations for i in range(n_rotations)]
    scales    = np.linspace(0.95, 1.05, n_scales).tolist()
    tta_variants = [(r, s) for r in rotations for s in scales]
    n_passes = len(tta_variants)

    # Val tiles
    ref_ds = GroundDataset(cfg,'val',augment=False)
    n_tiles = len(ref_ds)

    print(f'Geometric TTA [{label}]: {n_passes} passes '
          f'({n_rotations} rotations × {n_scales} scales) × {n_tiles} tiles...')

    all_probs  = []
    all_labels = []

    for tile_idx in tqdm(range(n_tiles), desc=f'GeoTTA {label}', ncols=88):
        pts_base, lbl = ref_ds[tile_idx]
        N = pts_base.shape[0]
        tile_probs = np.zeros(N, dtype=np.float32)

        for angle, scale in tta_variants:
            # Build dataset with this geometric variant
            tta_ds = GroundDataset(cfg,'val',augment=False,
                                   tta_angle=angle, tta_scale=scale)
            pts_t, _ = tta_ds[tile_idx]

            tensor = pts_t.unsqueeze(0).to(device)
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = m(tensor)
                p = F.softmax(logits[0],-1)[:,1].cpu().numpy()
            tile_probs += p
            del tensor, logits, p

        tile_probs /= n_passes
        all_probs.append(tile_probs)
        all_labels.append(lbl.numpy())

    probs = np.concatenate(all_probs)
    labs  = np.concatenate(all_labels)

    best_thr,best_f1,best_acc = 0.5,0.,0.; curve=[]
    for thr in np.arange(0.10,0.90,0.01):
        p=(probs>=thr).astype(np.int64)
        tp=((p==1)&(labs==1)).sum(); fp=((p==1)&(labs==0)).sum()
        fn=((p==0)&(labs==1)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
        f1=2*pr*rc/(pr+rc+1e-9); acc=(p==labs).mean()
        curve.append((float(thr),float(f1),float(acc)))
        if f1 > best_f1: best_f1,best_thr,best_acc=f1,float(thr),float(acc)

    # Explicit teardown — never hold two models on GPU simultaneously
    del m; torch.cuda.empty_cache(); gc.collect()
    return dict(model_path=model_path, threshold=best_thr,
                val_accuracy=best_acc, val_f1=best_f1, curve=curve,
                tta_passes=n_passes, tta_type='geometric')


# OOM safety: clear any residual GPU state before TTA sweep
# (Cell 8's GPU teardown already freed most memory; this is a belt-and-suspenders clear)
torch.cuda.empty_cache(); gc.collect()
for i in range(N_GPUS):
    used = torch.cuda.memory_allocated(i) / 1e6
    print(f'GPU {i} before TTA sweep: {used:.1f} MB')

# For time budget awareness
elapsed_min = (time.time()-SESSION_START)/60
print(f'Elapsed: {elapsed_min:.0f} min')

results = {}
N_ROT = 4   # 0, 90, 180, 270
N_SCL = 3   # 0.95, 1.00, 1.05  → 12 total passes

for lbl, path in [('best_epoch+GeoTTA', CFG['model_save']),
                   ('swa+GeoTTA',        CFG['swa_save'])]:
    r = sweep_geometric_tta(path, CFG, DEVICE, lbl,
                             n_rotations=N_ROT, n_scales=N_SCL)
    if r:
        results[lbl] = r
        print(f'[{lbl}]  thr={r["threshold"]:.2f}  '
              f'acc={r["val_accuracy"]*100:.2f}%  f1={r["val_f1"]:.4f}  '
              f'passes={r["tta_passes"]}')

if not results:
    print('No models found — run Cell 8 first'); raise SystemExit()

winner = max(results, key=lambda k: results[k]['val_accuracy'])
best   = results[winner]
print(f'\nWinner: {winner}  acc={best["val_accuracy"]*100:.2f}%')

# Plot
c   = best['curve']
fig,ax = plt.subplots(figsize=(8,4))
ax.plot([x[0] for x in c],[x[1] for x in c],label='F1')
ax.plot([x[0] for x in c],[x[2] for x in c],label='Accuracy')
ax.axvline(best['threshold'],ls='--',c='red',label=f'thr={best["threshold"]:.2f}')
ax.axhline(0.97,ls=':',c='orange',alpha=0.7,label='97%')
ax.axhline(0.98,ls=':',c='green', alpha=0.7,label='98%')
ax.set(xlabel='Threshold',ylim=(0.88,1.0),
       title=f'Threshold sweep — {winner} (Geometric TTA)')
ax.legend(); plt.tight_layout()
plt.savefig(str(Path(CFG['logs_dir'])/'threshold_curve.png'),dpi=120)
plt.close()

# Save metadata
meta = dict(
    model=winner, model_path=best['model_path'],
    threshold=best['threshold'], val_accuracy=best['val_accuracy'],
    val_f1=best['val_f1'], tta_passes=best['tta_passes'],
    tta_type='geometric', n_rotations=N_ROT, n_scales=N_SCL)
with open(CFG['threshold'],'w') as f: json.dump(meta,f,indent=2)

# Package
files = [best['model_path'], CFG['model_save'], CFG['swa_save'],
         CFG['threshold'], CFG['history'], CFG['curves'],
         str(Path(CFG['logs_dir'])/'threshold_curve.png')]

out_zip = Path('/kaggle/working/dtm_outputs_finetuned.zip')
with zipfile.ZipFile(str(out_zip),'w',zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        if Path(p).exists():
            zf.write(p, Path(p).name)
            print(f'  packed {Path(p).name} ({Path(p).stat().st_size/1e3:.0f} KB)')

print(f'\ndtm_outputs_finetuned.zip: {out_zip.stat().st_size/1e6:.1f} MB')

prev_acc = 0.9397  # Round 3 result
gain = best['val_accuracy'] - prev_acc

print(f'\nFINAL RESULT')
print(f'  Model         : {winner}')
print(f'  Threshold     : {best["threshold"]:.2f}')
print(f'  TTA passes    : {best["tta_passes"]} geometric (4 rot × 3 scale)')
print(f'  Accuracy      : {best["val_accuracy"]*100:.2f}%')
print(f'  F1 (ground)   : {best["val_f1"]:.4f}')
print(f'  Prev. best    : {prev_acc*100:.2f}% (Round 3)')
print(f'  Gain          : +{gain*100:.2f}pp')
if best['val_accuracy'] >= 0.98:
    print('  TARGET 98%+   : ✓ HIT')
elif best['val_accuracy'] >= 0.97:
    print('  TARGET 97%+   : ✓ HIT  (98% needs one more round)')
elif best['val_accuracy'] >= 0.95:
    print(f'  Gap to 98%    : {(0.98-best["val_accuracy"])*100:.2f}pp')
else:
    print(f'  Gap to 98%    : {(0.98-best["val_accuracy"])*100:.2f}pp — re-run Cell 8')
